In [7]:
from pathlib import Path
import json
import numpy as np

PROJECT_DIR = Path.cwd()
FEATURE_DIR = PROJECT_DIR / "features"
LABEL_FILE = PROJECT_DIR / "Labels.json"

print("Project directory:", PROJECT_DIR)
print("Labels path:", LABEL_FILE)
print("Features path:", FEATURE_DIR)

with LABEL_FILE.open("r", encoding="utf-8-sig") as file:
    labels = json.load(file)

print("Number of labels:", len(labels))

missing = []

for item in labels:
    npz_path = FEATURE_DIR / item["npz_file"]
    if not npz_path.is_file():
        missing.append(str(npz_path))

print("Missing NPZ files:", len(missing))

if missing:
    print(*missing[:10], sep="\n")
else:
    print("All NPZ files were found.")

Project directory: c:\Users\User\Documents\Push-Github\sign-language\rewrite_Phase3
Labels path: c:\Users\User\Documents\Push-Github\sign-language\rewrite_Phase3\Labels.json
Features path: c:\Users\User\Documents\Push-Github\sign-language\rewrite_Phase3\features
Number of labels: 227
Missing NPZ files: 0
All NPZ files were found.


In [8]:
required_arrays = {
    "hands",
    "pose",
    "hand_mask",
    "pose_mask",
    "pose_confidence",
}

frame_lengths = []
text_lengths = []
errors = []

for item in labels:
    npz_path = FEATURE_DIR / item["npz_file"]

    try:
        with np.load(npz_path) as data:
            missing_keys = required_arrays - set(data.files)
            if missing_keys:
                raise ValueError(f"Missing arrays: {missing_keys}")

            number_of_frames = data["hands"].shape[0]

            if data["hands"].shape != (number_of_frames, 2, 21, 3):
                raise ValueError(f"Invalid hands shape: {data['hands'].shape}")

            if data["pose"].shape != (number_of_frames, 9, 2):
                raise ValueError(f"Invalid pose shape: {data['pose'].shape}")

            if data["hand_mask"].shape != (number_of_frames, 2, 21):
                raise ValueError(
                    f"Invalid hand_mask shape: {data['hand_mask'].shape}"
                )

            if data["pose_mask"].shape != (number_of_frames, 9):
                raise ValueError(
                    f"Invalid pose_mask shape: {data['pose_mask'].shape}"
                )

            if data["pose_confidence"].shape != (number_of_frames, 9):
                raise ValueError(
                    "Invalid pose_confidence shape: "
                    f"{data['pose_confidence'].shape}"
                )

            frame_lengths.append(number_of_frames)
            text_lengths.append(len(item["text"]))

    except Exception as error:
        errors.append((item["npz_file"], str(error)))

print("Valid samples:", len(frame_lengths))
print("Errors:", len(errors))

if frame_lengths:
    print("Frame length:")
    print("  min =", min(frame_lengths))
    print("  max =", max(frame_lengths))
    print("  mean =", round(np.mean(frame_lengths), 2))

    print("Text length:")
    print("  min =", min(text_lengths))
    print("  max =", max(text_lengths))
    print("  mean =", round(np.mean(text_lengths), 2))

if errors:
    print("First errors:")
    for error in errors[:10]:
        print(error)

Valid samples: 227
Errors: 0
Frame length:
  min = 512
  max = 7563
  mean = 1781.44
Text length:
  min = 53
  max = 424
  mean = 146.32


In [10]:
for percentile in [50, 75, 90, 95, 99, 100]:
    value = np.percentile(frame_lengths, percentile)
    print(f"P{percentile}: {value:.0f} frames")

print()

for factor in [2, 3, 4]:
    reduced = np.ceil(np.array(frame_lengths) / factor)
    print(
        f"Downsample ×{factor}: "
        f"mean={reduced.mean():.0f}, "
        f"max={reduced.max():.0f}"
    )

P50: 1290 frames
P75: 2106 frames
P90: 3467 frames
P95: 3910 frames
P99: 6952 frames
P100: 7563 frames

Downsample ×2: mean=891, max=3782
Downsample ×3: mean=594, max=2521
Downsample ×4: mean=446, max=1891


In [11]:
from pathlib import Path
import json
import numpy as np

DOWNSAMPLE_FACTOR = 4
PROCESSED_DIR = PROJECT_DIR / "processed_features"
MANIFEST_FILE = PROJECT_DIR / "manifest.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def normalize_hand(points, point_mask):
    """
    points:     (21, 3)
    point_mask: (21,)
    """
    output = np.zeros((21, 3), dtype=np.float32)

    # Normalization requires both the wrist and landmark 9.
    if not point_mask[0] or not point_mask[9]:
        return output

    wrist = points[0]
    scale = np.linalg.norm(points[9] - wrist)

    if not np.isfinite(scale) or scale < 1e-6:
        return output

    output = (points - wrist) / scale
    output[~point_mask.astype(bool)] = 0.0

    return output.astype(np.float32)


def convert_npz_to_features(npz_path, downsample_factor=4):
    with np.load(npz_path) as data:
        hands = data["hands"].astype(np.float32)
        hand_mask = data["hand_mask"].astype(bool)

    number_of_frames = hands.shape[0]
    normalized_hands = np.zeros_like(hands, dtype=np.float32)

    # One presence value for each hand.
    hand_present = hand_mask.any(axis=2).astype(np.float32)

    for frame_index in range(number_of_frames):
        for hand_index in range(2):
            if hand_present[frame_index, hand_index]:
                normalized_hands[frame_index, hand_index] = normalize_hand(
                    hands[frame_index, hand_index],
                    hand_mask[frame_index, hand_index],
                )

    # 2 hands × 21 landmarks × 3 coordinates = 126
    hand_coordinates = normalized_hands.reshape(number_of_frames, 126)

    # 126 coordinates + 2 presence values = 128
    features = np.concatenate(
        [hand_coordinates, hand_present],
        axis=1,
    ).astype(np.float32)

    features = features[::downsample_factor]

    return features


manifest = []

for item in labels:
    npz_path = FEATURE_DIR / item["npz_file"]
    output_name = Path(item["npz_file"]).with_suffix(".npy").name
    output_path = PROCESSED_DIR / output_name

    features = convert_npz_to_features(
        npz_path,
        downsample_factor=DOWNSAMPLE_FACTOR,
    )

    np.save(output_path, features)

    manifest_item = {
        "clip_id": Path(item["npz_file"]).stem,
        "npy_path": output_path.relative_to(PROJECT_DIR).as_posix(),
        "text": item["text"],
        "num_frames": int(features.shape[0]),
    }

    # Preserve source-video information if Labels.json contains it.
    if "source_video" in item:
        manifest_item["source_video"] = item["source_video"]

    if "start_second" in item:
        manifest_item["start_second"] = item["start_second"]

    if "end_second" in item:
        manifest_item["end_second"] = item["end_second"]

    manifest.append(manifest_item)


with MANIFEST_FILE.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

print("Processed clips:", len(manifest))
print("Manifest:", MANIFEST_FILE)
print("First feature shape:", np.load(
    PROJECT_DIR / manifest[0]["npy_path"]
).shape)

processed_lengths = [item["num_frames"] for item in manifest]

print("Minimum length:", min(processed_lengths))
print("Maximum length:", max(processed_lengths))
print("Average length:", round(np.mean(processed_lengths), 2))

Processed clips: 227
Manifest: c:\Users\User\Documents\Push-Github\sign-language\rewrite_Phase3\manifest.json
First feature shape: (368, 128)
Minimum length: 128
Maximum length: 1891
Average length: 445.74


In [12]:
print(json.dumps(manifest[0], ensure_ascii=False, indent=2))

missing_source = [
    item["clip_id"]
    for item in manifest
    if not item.get("source_video")
]

print("Clips missing source_video:", len(missing_source))
print(*missing_source[:10], sep="\n")

{
  "clip_id": "part_1_1_upper_body_coordinates",
  "npy_path": "processed_features/part_1_1_upper_body_coordinates.npy",
  "text": "看新聞 開視野 歡迎收看今天6月6號的公視手語新聞 首先來關心臺南市永康鬧區 昨晚突然飄出瓦斯異位 臺南市警消貨報到場偵測到可燃性氣體 緊急疏散周邊店家人員 封路不水線防護 瓦斯公司也關閉閥門 尋找洩漏員 由於今天是上班日 要看搶修進度才能確認 是否影響白天上班通勤",
  "num_frames": 368
}
Clips missing source_video: 227
part_1_1_upper_body_coordinates
part_1_2_upper_body_coordinates
part_1_3_upper_body_coordinates
part_1_5_upper_body_coordinates
part_1_7_upper_body_coordinates
part_2_1_upper_body_coordinates
part_2_3_upper_body_coordinates
part_2_6_upper_body_coordinates
part_2_7_upper_body_coordinates
part_2_8_upper_body_coordinates
